# OptiMystic JupyterLab Decision Demo

This notebook solves one base business problem first, then runs a compact sensitivity analysis.

## Base problem (plain language)
- A team must choose which components to load into a limited-capacity shipment.
- Each component has a weight and a value.
- Goal: maximize total value without exceeding capacity.

So this notebook is not about math theory first. It is about a practical question:
"Given today's constraints, which mix of components should we actually load?"

## Why sensitivity analysis is included
After solving the base case, we vary one factor at a time (capacity, value level, weight pressure) to check how stable the recommendation is.

In [1]:
# Step 1: Environment bootstrap
import json
import sys
import importlib
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "examples").exists() and (PROJECT_ROOT.parent / "examples").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "examples"))
import jupyter_debug_tools as jdt
importlib.reload(jdt)

run_python_only_cp_scheduling = jdt.run_python_only_cp_scheduling
run_julia_only_mip_packing = jdt.run_julia_only_mip_packing
run_full_pipeline = jdt.run_full_pipeline
ensure_r_bridge = jdt.ensure_r_bridge
run_r_postprocess = jdt.run_r_postprocess
explain_debug_pipeline = jdt.explain_debug_pipeline
run_section = jdt.run_section
warmup_julia_path = jdt.warmup_julia_path

print("Project root:", PROJECT_ROOT)
print("Loaded helper module from:", jdt.__file__)
print(json.dumps(explain_debug_pipeline(), indent=2))

Project root: c:\Projects\OptiMystic
Loaded helper module from: c:\Projects\OptiMystic\examples\jupyter_debug_tools.py
{
  "python_only": "Validate only Python CP scheduling logic (excluding Julia/R)",
  "julia_only": "Validate only Julia MIP execution path (called via Python CLI subprocess)",
  "r_bridge": "Validate rpy2 and r_solvers loading/connectivity",
  "full_pipeline": "End-to-end validation: Python/Julia outputs passed into R process_results",
  "quick_mode": "Use quick=True to run smaller payloads for faster feedback",
  "warmup": "Run warmup_julia_path() once to reduce first-run Julia latency"
}


## Notebook Flow

- Cell 1-3: environment and runtime checks
- Cell 6: solve one base case + generate one-factor sensitivity cases
- Cell 7: decision dashboard with a spider chart for sensitivity interpretation

Key interpretation fields:
- objective: total business value achieved
- utilization: how fully capacity is used
- selected_items: which items are actually selected
- delta_vs_base: how much each sensitivity case moves from the base result

In [2]:
# Step 2: Fast Python/Julia checks and reusable helpers
from pprint import pprint

print("Warming up Julia path (first run can be slower)...")
warmup = warmup_julia_path()
print(json.dumps(warmup, indent=2))

# Quick targeted run
SECTION = "full"  # "python" | "julia" | "full"
section_result = run_section(SECTION, quick=True)
print(f"Section={SECTION}")
print(json.dumps({k: v.get('status') for k, v in section_result.items()}, indent=2))


def solve_packing_case(capacity, items, scenario_name):
    params = {"Items": items, "Vehicles": [{"Capacity": capacity}]}
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "python_solvers" / "cli_solver.py"),
        "--domain",
        "packing",
        "--solver",
        "mip",
        "--params",
        json.dumps(params),
    ]
    proc = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd=str(PROJECT_ROOT),
        timeout=180,
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or proc.stdout.strip() or f"Packing solve failed for {scenario_name}")

    solved = json.loads(proc.stdout)
    store = {
        "parameters": {
            "Items": [item["Name"] for item in items],
            "Weights": [item["Weight"] for item in items],
            "Values": [item["Value"] for item in items],
            "Capacity": capacity,
            "Scenario": scenario_name,
        }
    }
    processed = run_r_postprocess("packing", solved, store)
    return solved, processed, store


def run_r_decision_analytics(run_results, mode="packing"):
    ro = importlib.import_module("rpy2.robjects")
    runs_json = json.dumps(run_results)
    ro.globalenv["py_runs_json"] = runs_json
    ro.r(f"py_analytics <- process_decision_analytics(jsonlite::fromJSON(py_runs_json, simplifyVector = FALSE), mode = '{mode}')")
    analytics_json = str(ro.r("jsonlite::toJSON(py_analytics, auto_unbox = TRUE, null = 'null')")[0])
    return json.loads(analytics_json)


def run_r_variant_compare(run_results, variant_labels, mode="packing"):
    ro = importlib.import_module("rpy2.robjects")
    ro.globalenv["py_runs_json"] = json.dumps(run_results)
    ro.globalenv["py_labels_json"] = json.dumps(variant_labels)
    ro.r("py_runs <- jsonlite::fromJSON(py_runs_json, simplifyVector = FALSE)")
    ro.r("py_labels <- unlist(jsonlite::fromJSON(py_labels_json, simplifyVector = TRUE))")
    ro.r(f"py_compare <- compare_solver_performance(py_runs, py_labels, mode = '{mode}')")
    compare_json = str(ro.r("jsonlite::toJSON(py_compare, auto_unbox = TRUE, null = 'null')")[0])
    return json.loads(compare_json)


def run_r_summary(processed_result, decision_analytics=None, sensitivity=None):
    ro = importlib.import_module("rpy2.robjects")
    ro.globalenv["py_processed_json"] = json.dumps(processed_result)
    ro.r("py_processed <- jsonlite::fromJSON(py_processed_json, simplifyVector = FALSE)")
    ro.r("py_sensitivity <- NULL")
    if sensitivity is not None:
        ro.globalenv["py_sensitivity_json"] = json.dumps(sensitivity)
        ro.r("py_sensitivity <- jsonlite::fromJSON(py_sensitivity_json, simplifyVector = FALSE)")
    ro.r("py_analytics <- NULL")
    if decision_analytics is not None:
        ro.globalenv["py_analytics_json"] = json.dumps(decision_analytics)
        ro.r("py_analytics <- jsonlite::fromJSON(py_analytics_json, simplifyVector = FALSE)")
    summary_text = ro.r("build_executive_summary(py_processed, py_sensitivity, py_analytics)")
    return str(summary_text[0])

Warming up Julia path (first run can be slower)...
{
  "status": "Optimal",
  "objective": 10.0,
  "note": "Julia path warmed up. Next runs are usually faster."
}
Section=full
{
  "python_cp": "Optimal",
  "julia_mip": "Optimal"
}


In [3]:
# Step 3: R bridge check
r_info = ensure_r_bridge()
print(json.dumps(r_info, indent=2))

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb4 in position 1: invalid start byte <traceback object at 0x000002234A08C6C0>
R callback write-console: The following objects are masked from 'package:stats':

    filter, lag

  
R callback write-console: The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

  


{
  "ok": true,
  "r_version": "R version 4.5.3 (2026-03-11 ucrt)",
  "r_workdir": "C:/Projects/OptiMystic/r_solvers"
}


## Decision questions answered here

1. What is the best item mix for the base operating condition?
2. If capacity is tighter or looser, does the recommended mix change a lot?
3. If value drops or weight pressure increases, how much performance is lost?
4. Which scenarios are robust, and which are fragile compared to the base case?

In [8]:
# Step 4: Solve one base case, then run one-factor sensitivity analysis
base_items = [
    {"Name": "Battery Pack", "Weight": 2, "Value": 6},
    {"Name": "Sensor Module", "Weight": 3, "Value": 9},
    {"Name": "Control Unit", "Weight": 4, "Value": 12},
    {"Name": "Pump Assembly", "Weight": 5, "Value": 15},
    {"Name": "Valve Kit", "Weight": 6, "Value": 17},
    {"Name": "Frame", "Weight": 7, "Value": 20},
    {"Name": "Cooling Plate", "Weight": 8, "Value": 22},
    {"Name": "Cable Harness", "Weight": 2, "Value": 7},
    {"Name": "Mount Set", "Weight": 1, "Value": 4},
    {"Name": "Service Kit", "Weight": 3, "Value": 10},
]

base_capacity = 16

sensitivity_definitions = [
    {"name": "base", "capacity": base_capacity, "weight_mult": 1.00, "value_mult": 1.00},
    {"name": "capacity_tight", "capacity": int(round(base_capacity * 0.80)), "weight_mult": 1.00, "value_mult": 1.00},
    {"name": "capacity_roomy", "capacity": int(round(base_capacity * 1.20)), "weight_mult": 1.00, "value_mult": 1.00},
    {"name": "value_down", "capacity": base_capacity, "weight_mult": 1.00, "value_mult": 0.85},
    {"name": "value_up", "capacity": base_capacity, "weight_mult": 1.00, "value_mult": 1.15},
    {"name": "weight_pressure", "capacity": base_capacity, "weight_mult": 1.20, "value_mult": 1.00},
]


def build_items_for_case(items, case_def):
    built = []
    for item in items:
        built.append(
            {
                "Name": item["Name"],
                "Weight": max(1, int(round(item["Weight"] * case_def["weight_mult"]))),
                "Value": max(1, int(round(item["Value"] * case_def["value_mult"]))),
            }
        )
    return built


scenario_runs = []
scenario_processed = []
scenario_rows = []

for case_def in sensitivity_definitions:
    case_items = build_items_for_case(base_items, case_def)
    solved, processed, _ = solve_packing_case(case_def["capacity"], case_items, case_def["name"])

    scenario_runs.append(solved)
    scenario_processed.append(processed)

    selected_items = [item.get("item") for item in processed.get("items", [])]
    used_capacity = float(processed.get("used_capacity", 0) or 0)
    objective = float(processed.get("total_value", 0) or 0)

    scenario_rows.append(
        {
            "scenario_name": case_def["name"],
            "capacity": case_def["capacity"],
            "weight_mult": case_def["weight_mult"],
            "value_mult": case_def["value_mult"],
            "status": processed.get("status"),
            "objective": round(objective, 2),
            "used_capacity": round(used_capacity, 2),
            "utilization_pct": round(100.0 * used_capacity / max(case_def["capacity"], 1), 1),
            "selected_count": len(selected_items),
            "selected_items": selected_items,
            "selected_items_str": ", ".join(selected_items) if selected_items else "none",
        }
    )

base_row = next(row for row in scenario_rows if row["scenario_name"] == "base")
base_objective = float(base_row["objective"])
base_utilization = float(base_row["utilization_pct"])
base_selected_count = int(base_row["selected_count"])

for row in scenario_rows:
    row["delta_objective_vs_base"] = round(float(row["objective"]) - base_objective, 2)
    row["delta_utilization_vs_base"] = round(float(row["utilization_pct"]) - base_utilization, 2)
    row["delta_selected_count_vs_base"] = int(row["selected_count"] - base_selected_count)

decision_analytics = run_r_decision_analytics(scenario_runs, mode="packing")
best_index = max(range(len(scenario_processed)), key=lambda idx: float(scenario_processed[idx].get("total_value", 0) or 0))
best_processed = scenario_processed[best_index]
best_scenario_name = scenario_rows[best_index]["scenario_name"]
best_capacity = scenario_rows[best_index]["capacity"]
executive_summary = run_r_summary(best_processed, decision_analytics=decision_analytics)

print("[Base case]")
print(json.dumps(base_row, indent=2))
print("\n[Sensitivity results]")
pprint(scenario_rows)
print("\n[Decision analytics]")
print(json.dumps(decision_analytics, indent=2))
print("\n[Best scenario]")
print(
    json.dumps(
        {
            "scenario_name": best_scenario_name,
            "capacity": best_capacity,
            "objective": best_processed.get("total_value"),
            "used_capacity": best_processed.get("used_capacity"),
            "items": [item.get("item") for item in best_processed.get("items", [])],
        },
        indent=2,
    )
)
print("\n[Executive summary]")
print(executive_summary)

[Base case]
{
  "scenario_name": "base",
  "capacity": 16,
  "weight_mult": 1.0,
  "value_mult": 1.0,
  "status": "ok",
  "objective": 51.0,
  "used_capacity": 16.0,
  "utilization_pct": 100.0,
  "selected_count": 6,
  "selected_items": [
    "Battery Pack",
    "Sensor Module",
    "Pump Assembly",
    "Cable Harness",
    "Mount Set",
    "Service Kit"
  ],
  "selected_items_str": "Battery Pack, Sensor Module, Pump Assembly, Cable Harness, Mount Set, Service Kit",
  "delta_objective_vs_base": 0.0,
  "delta_utilization_vs_base": 0.0,
  "delta_selected_count_vs_base": 0
}

[Sensitivity results]
[{'capacity': 16,
  'delta_objective_vs_base': 0.0,
  'delta_selected_count_vs_base': 0,
  'delta_utilization_vs_base': 0.0,
  'objective': 51.0,
  'scenario_name': 'base',
  'selected_count': 6,
  'selected_items': ['Battery Pack',
                     'Sensor Module',
                     'Pump Assembly',
                     'Cable Harness',
                     'Mount Set',
                 

In [9]:
# Step 5: Sensitivity dashboard with spider chart
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

scenario_df = pd.DataFrame(scenario_rows)
scenario_df = scenario_df.sort_values(["objective", "scenario_name"], ascending=[False, True]).reset_index(drop=True)

objective_ci = decision_analytics.get("objective_ci", {})
solve_time_ci = decision_analytics.get("solve_time_ci", {})
print("[Dashboard summary]")
print(executive_summary)
print(
    json.dumps(
        {
            "feasible_rate_pct": round(100.0 * float(decision_analytics.get("feasible_rate", 0) or 0), 1),
            "objective_ci": {
                "mean": objective_ci.get("mean"),
                "lower": objective_ci.get("lower"),
                "upper": objective_ci.get("upper"),
            },
            "solve_time_ci": {
                "mean": solve_time_ci.get("mean"),
                "lower": solve_time_ci.get("lower"),
                "upper": solve_time_ci.get("upper"),
            },
        },
        indent=2,
    )
)

print("\n[Sensitivity table]")
display(
    scenario_df[
        [
            "scenario_name",
            "capacity",
            "weight_mult",
            "value_mult",
            "objective",
            "delta_objective_vs_base",
            "utilization_pct",
            "delta_utilization_vs_base",
            "selected_count",
            "delta_selected_count_vs_base",
            "status",
        ]
    ]
)

# Spider chart metrics (normalized around base=100 for direct sensitivity interpretation)
base_row_df = scenario_df[scenario_df["scenario_name"] == "base"].iloc[0]
base_objective = max(1.0, float(base_row_df["objective"]))
base_utilization = max(1.0, float(base_row_df["utilization_pct"]))
base_selected_count = max(1.0, float(base_row_df["selected_count"]))
base_value_density = max(1e-6, float(base_row_df["objective"]) / max(float(base_row_df["used_capacity"]), 1.0))

radar_categories = [
    "objective_index",
    "utilization_index",
    "selected_count_index",
    "value_density_index",
]
radar_labels = {
    "objective_index": "Objective",
    "utilization_index": "Utilization",
    "selected_count_index": "Selected Count",
    "value_density_index": "Value Density",
}

radar_df = scenario_df.copy()
radar_df["objective_index"] = 100.0 * radar_df["objective"] / base_objective
radar_df["utilization_index"] = 100.0 * radar_df["utilization_pct"] / base_utilization
radar_df["selected_count_index"] = 100.0 * radar_df["selected_count"] / base_selected_count
radar_df["value_density_index"] = 100.0 * (
    (radar_df["objective"] / radar_df["used_capacity"].clip(lower=1.0)) / base_value_density
)

fig_radar = go.Figure()
for _, row in radar_df.iterrows():
    fig_radar.add_trace(
        go.Scatterpolar(
            r=[float(row[c]) for c in radar_categories] + [float(row[radar_categories[0]])],
            theta=[radar_labels[c] for c in radar_categories] + [radar_labels[radar_categories[0]]],
            fill="toself",
            name=str(row["scenario_name"]),
            opacity=0.45 if row["scenario_name"] != "base" else 0.75,
            hovertemplate=(
                "scenario=%{text}<br>metric=%{theta}<br>index=%{r:.1f}<extra></extra>"
            ),
            text=[str(row["scenario_name"])] * (len(radar_categories) + 1),
        )
    )

fig_radar.update_layout(
    title="Sensitivity Spider Chart (Base = 100)",
    template="plotly_white",
    height=620,
    polar=dict(radialaxis=dict(visible=True, range=[0, max(130, float(radar_df[radar_categories].max().max()) + 10)])),
)
fig_radar.show()

# Delta chart for quick business interpretation
fig_delta = px.bar(
    scenario_df,
    x="scenario_name",
    y="delta_objective_vs_base",
    color="delta_objective_vs_base",
    text="delta_objective_vs_base",
    title="Objective Delta vs Base Scenario",
    color_continuous_scale="RdYlGn",
)
fig_delta.update_layout(template="plotly_white", height=480, xaxis_title="scenario_name", yaxis_title="objective delta")
fig_delta.show()

best_items_df = pd.DataFrame(best_processed.get("items", []))
if not best_items_df.empty:
    best_items_df = best_items_df.sort_values("value", ascending=True)
    fig_best = px.bar(
        best_items_df,
        x="value",
        y="item",
        color="weight",
        orientation="h",
        text="count",
        title=f"Best Scenario Item Mix ({best_scenario_name}, capacity={best_capacity})",
        color_continuous_scale="Viridis",
    )
    fig_best.update_layout(template="plotly_white", height=520)
    fig_best.show()
else:
    print("No selected items in best scenario.")

[Dashboard summary]
Mode: packing
Status: ok
Primary objective metric: 59.0000
Observed runs: 6
Feasible rate: 100.00%
Decision recommendation: Runs are feasible but unstable outliers exist; inspect seeds, time limits, and scenario mix.
{
  "feasible_rate_pct": 100.0,
  "objective_ci": {
    "mean": 49.6667,
    "lower": 43.6667,
    "upper": 55.1667
  },
  "solve_time_ci": {
    "mean": 3.0133,
    "lower": 2.8375,
    "upper": 3.2643
  }
}

[Sensitivity table]


,scenario_name,capacity,weight_mult,value_mult,objective,delta_objective_vs_base,utilization_pct,delta_utilization_vs_base,selected_count,delta_selected_count_vs_base,status
0,capacity_roomy,19,1.0,1.00,59.0,8.0,100.0,0.0,6,0,ok
1,value_up,16,1.0,1.15,59.0,8.0,100.0,0.0,5,-1,ok
2,base,16,1.0,1.00,51.0,0.0,100.0,0.0,6,0,ok
3,weight_pressure,16,1.2,1.00,44.0,-7.0,100.0,0.0,5,-1,ok
4,value_down,16,1.0,0.85,43.0,-8.0,100.0,0.0,6,0,ok
5,capacity_tight,13,1.0,1.00,42.0,-9.0,100.0,0.0,5,-1,ok


## Phase 2.5 Mock Test (LLM Role-Play Rehearsal)

The cell below is a final rehearsal where the developer acts like an LLM agent and executes the MCP tool chain step by step.

Flow:
1. Use `read_company_data` to inspect the source file.
2. Use `get_target_schema('packing')` to check target requirements.
3. Write a mapping rule manually.
4. Use `map_to_target_schema` for transformation + Pydantic validation.
5. Run `optimize` and confirm `Optimal` status.

In [ ]:
# Phase 2.5 Step 1-5: Manual Developer Mock Pipeline
import asyncio
import json
from python_solvers.mcp_server import mcp


async def call_tool(name, args):
    result = await mcp._tool_manager.call_tool(name, args)
    return result.structured_content


sample_path = "examples/sample.csv"

# 1) Source data inspection
read_result = asyncio.run(call_tool("read_company_data", {"file_path": sample_path, "max_rows": 3}))
print("[1] read_company_data")
print(json.dumps(read_result, ensure_ascii=False, indent=2))

# 2) Target schema check
schema_result = asyncio.run(call_tool("get_target_schema", {"domain": "packing"}))
print("\n[2] get_target_schema('packing')")
print(json.dumps({"ok": schema_result.get("ok"), "domain": schema_result.get("domain")}, ensure_ascii=False, indent=2))

# 3) Manual mapping rule authored by developer
mapping_rule = {
    "품목명": "Name",
    "단위중량(kg)": "Weight",
    "당일발주량": "Demand",
}
print("\n[3] mapping_rule")
print(json.dumps(mapping_rule, ensure_ascii=False, indent=2))

# 4) Transformation + Pydantic validation
map_result = asyncio.run(
    call_tool(
        "map_to_target_schema",
        {
            "file_path": sample_path,
            "mapping_rule": mapping_rule,
            "domain": "packing",
        },
    )
)
print("\n[4] map_to_target_schema")
print(json.dumps(map_result, ensure_ascii=False, indent=2))

# 5) Optimize call
optimize_result = asyncio.run(
    call_tool(
        "optimize",
        {
            "request": {
                "domain": "packing",
                "solver": "mip",
                "params": map_result.get("payload", {}),
            }
        },
    )
)

print("\n[5] optimize")
print(json.dumps(optimize_result, ensure_ascii=False, indent=2))

status = (optimize_result.get("result") or {}).get("status")
print("\nFinal status:", status)